In [3]:
import pandas as pd
import json
import requests
from dotenv import load_dotenv
import time
import os
from pathlib import Path
params = {"format": "json"}
load_dotenv("../.env")
key = os.environ['CONGRESS_API_KEY']
headers = {"X-API-Key": key}

In [4]:
bills = pd.read_csv("../data/processed_bills/bills.csv")
bills.head(1)



,congress,introducedDate,number,originChamber,originChamberCode,title,type,updateDate,updateDateIncludingText,url,label_latest_action_date,label_latest_action_text,label_latest_action_time,bill_id
0,118,2023-01-09,8,House,H,Proposing an amendment to the Constitution of ...,HJRES,2025-12-05,2025-12-05,https://api.congress.gov/v3/bill/118/hjres/8?f...,2023-01-09,Referred to the House Committee on the Judiciary.,NaN,118HJRES8


In [5]:
# sampling some rows 

test = bills.groupby("type").sample(n=5, random_state=42)


In [6]:
for bill in test.itertuples():
    filename = f"test_data/{bill.bill_id}.json"
    url = f"https://api.congress.gov/v3/bill/{bill.congress}/{bill.type}/{bill.number}"
    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    data = response.json()
    with open(filename, "w") as f:
        json.dump(data, f)
    time.sleep(1)

    print(bill.congress, bill.type)
    


118 HJRES
118 HJRES
118 HJRES
118 HJRES
118 HJRES
118 HR
118 HR
118 HR
118 HR
118 HR
118 S
118 S
118 S
118 S
118 S
118 SJRES
118 SJRES
118 SJRES
118 SJRES
118 SJRES


In [7]:
with open("test_data/118SJRES23.json", "r") as file:
    data = json.load(file)
data["bill"].keys()

dict_keys(['actions', 'committees', 'congress', 'cosponsors', 'introducedDate', 'latestAction', 'legislationUrl', 'number', 'originChamber', 'originChamberCode', 'policyArea', 'relatedBills', 'sponsors', 'subjects', 'summaries', 'textVersions', 'title', 'titles', 'type', 'updateDate', 'updateDateIncludingText'])

In [8]:
print(data["bill"].get("laws"))

None


In [9]:
print(data['bill']['sponsors'])

[{'bioguideId': 'L000571', 'firstName': 'Cynthia', 'fullName': 'Sen. Lummis, Cynthia M. [R-WY]', 'isByRequest': 'N', 'lastName': 'Lummis', 'middleName': 'M.', 'party': 'R', 'state': 'WY', 'url': 'https://api.congress.gov/v3/member/L000571?format=json'}]


# test run


In [10]:
rows = []
for file in sorted(Path("test_data").glob("*.json")):
    with open(file) as f:
        bill = json.load(f)['bill']

    sponsors = bill.get("sponsors", []) # [] if key is missing
    sponsor = sponsors[0] if sponsors else {} # {} if no sponsor

    rows.append({
        "bill_id": file.stem, # filename - .json
        "sponsor_bioguide_id": sponsor.get("bioguideId"), # none if missing
        "sponsor_party": sponsor.get("party"),
        "sponsor_state": sponsor.get("state")
    })

sponsor_df = pd.DataFrame(rows)
print(sponsor_df.shape)
print(sponsor_df["sponsor_bioguide_id"].isna().sum(), "bills with no sponsor")

(20, 4)
0 bills with no sponsor


In [11]:
# attaching sponsor to sample bills

pilot = test.merge(sponsor_df, on="bill_id", how="left", validate="one_to_one")
print(pilot.shape)

(20, 17)


In [22]:
# attaching voteview ideology

ideology = pd.read_csv("../data/ideology_data/congress_118_ideology.csv")
ideology = ideology[ideology['chamber'].isin(["House", "Senate"])]

# rename so both tables use same key name
ideology = ideology.rename(columns={"bioguide_id": "sponsor_bioguide_id"})
ideology = ideology[['sponsor_bioguide_id', 'congress', 'nominate_dim1', 'nominate_dim2']]

pilot = pilot.merge(
    ideology, 
    on=['sponsor_bioguide_id', 'congress'],
    how='left',
    validate='many_to_one', # many bills per member, one member row per key
    indicator=True
)

print(pilot["_merge"].value_counts())
print(pilot[["bill_id", "sponsor_bioguide_id", "nominate_dim1", "_merge"]])


MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 sponsor_bioguide_id  congress
            S001150       118
            K000394       118
            M001183       118 ...

In [12]:
dupes = ideology[ideology["sponsor_bioguide_id"].isin(["S001150", "K000394", "M001183"])]
dupes

NameError: name 'ideology' is not defined

Duplicate entries are breaking the merge. Joe Manchin did this. 

In [39]:
ideology = ideology.drop_duplicates(subset=["sponsor_bioguide_id", "congress"], keep="first")
ideology

,sponsor_bioguide_id,congress,nominate_dim1,nominate_dim2
1,R000575,118,0.379,0.377
2,S001185,118,-0.402,0.397
3,P000609,118,0.670,0.113
4,C001054,118,0.550,0.239
5,M001212,118,0.646,-0.240
...,...,...,...,...
551,M001183,118,-0.057,0.400
553,B001230,118,-0.484,-0.123
554,J000293,118,0.648,-0.162
555,L000571,118,0.682,-0.311


In [37]:
# attaching voteview ideology
ideology = ideology[ideology['chamber'].isin(["House", "Senate"])]

# rename so both tables use same key name
ideology = ideology.rename(columns={"bioguide_id": "sponsor_bioguide_id"})
ideology = ideology[['sponsor_bioguide_id', 'congress', 'nominate_dim1', 'nominate_dim2']]

pilot = pilot.merge(
    ideology, 
    on=['sponsor_bioguide_id', 'congress'],
    how='left',
    validate='many_to_one', # many bills per member, one member row per key
    indicator=True
)

print(pilot["_merge"].value_counts())
print(pilot[["bill_id", "sponsor_bioguide_id", "nominate_dim1", "_merge"]])

KeyError: 'chamber'